In [ ]:
# Using json for structured data (like input/output) and potentially parsing feedback later if needed
import json
# To interact with the file system (reading data files) and environment variables (API keys)
import os
# Import the Google Generative AI library (Gemini)
import google.generativeai as genai
# Import specific types from the Gemini library for configuration and content handling
import google.generativeai.types as genai_types
# Import specific exceptions from google.api_core to catch API errors like quota limits
from google.api_core import exceptions
# To add delays between API calls, measure time for ETA
import time
# To add randomness (jitter) to backoff delays, preventing thundering herd issues
import random
# To help find option keys using regular expressions (though not used in the final version)
import re
# For calculating time differences easily
from datetime import timedelta

# --- Google Colab Specific Imports ---
# Attempt to import Colab specific modules
colab_env = False
try:
    from google.colab import drive
    from google.colab import userdata # Import userdata for secrets
    drive.mount('/content/drive', force_remount=True) # Remount drive
    print("Google Drive mounted successfully.")
    colab_env = True
    print("Google Colab environment detected. Will try reading secrets from userdata.")
except ImportError:
    print("Google Colab environment not detected or modules unavailable.")
    print("Will read API keys from standard OS environment variables.")
# --- --- --- --- --- --- --- --- --- ---

# --- Constants ---
MAX_API_KEYS = 6 # Maximum number of API keys to check for
MAX_RETRIES_PER_KEY = 5 # Max retries on a single key before switching
INITIAL_BACKOFF_DELAY = 10 # Initial delay in seconds for exponential backoff
BACKOFF_FACTOR = 2 # Factor to multiply delay by on each retry
ETA_UPDATE_INTERVAL_LINES = 10 # Update ETA every N lines processed
MIN_LINES_FOR_ETA = 5 # Minimum lines to process before calculating first ETA

# --- Global Variables for API Keys ---
# Use lists to store the keys and their names
api_keys = []
key_names = []
# Index of the currently active key in the api_keys list
current_key_index = 0
# Define the specific Gemini model name to be used by all agents globally
gemini_model_name = 'gemini-2.0-flash' # Keep using 1.5-flash unless 2.0 is confirmed

# --- Gemini Configuration ---
# IMPORTANT:
# If in Google Colab: Set your API keys using the 'Secrets' tab (key icon on the left).
# Name them: GEMINI_API_KEY_1, GEMINI_API_KEY_2, ..., GEMINI_API_KEY_6
# If not in Colab: Set them as standard OS environment variables with the same names.
# DO NOT HARDCODE YOUR API KEYS DIRECTLY IN THE CODE.
print("--- Loading API Keys ---")
keys_loaded_count = 0
for i in range(1, MAX_API_KEYS + 1):
    key_env_var = f"GEMINI_API_KEY_{i}"
    key = None
    # Try reading from Colab userdata first if available
    if colab_env:
        try:
            key = userdata.get(key_env_var)
            if key:
                 print(f"Found {key_env_var} in Colab Secrets.")
            # else:
            #      print(f"{key_env_var} not found in Colab Secrets.") # Can be verbose
        except userdata.SecretNotFoundError:
             # print(f"{key_env_var} not found in Colab Secrets.") # Can be verbose
             pass
        except Exception as e:
            print(f"Warning: Error reading {key_env_var} from Colab userdata: {e}")

    # If not found in userdata or not in Colab, try environment variables
    if not key:
        key = os.environ.get(key_env_var)
        if key:
            print(f"Found {key_env_var} in OS Environment Variables.")
        # else:
        #     print(f"{key_env_var} not found in OS Environment Variables.") # Can be verbose

    # If key was found by either method, add it
    if key:
        api_keys.append(key)
        key_names.append(f"Key_{i}")
        keys_loaded_count += 1
    # else:
    #     print(f"{key_env_var} not found.") # Can be verbose

if keys_loaded_count == 0:
    print("\nCRITICAL WARNING: No GEMINI_API_KEY environment variables/secrets found (checked GEMINI_API_KEY_1 to GEMINI_API_KEY_6).")
    print("Please add keys to Colab Secrets (if applicable) or set OS environment variables.")
    print("API calls WILL fail.")
else:
    print(f"\nLoaded {keys_loaded_count} API key(s).")
    try:
        # Configure with the first available key
        current_key_index = 0
        print(f"Configuring Gemini API with initial key: {key_names[current_key_index]} using model {gemini_model_name}.")
        genai.configure(api_key=api_keys[current_key_index])
        print("Gemini API configured successfully.")
    except IndexError:
         print("Error: Attempted to configure Gemini, but the api_keys list appears empty despite reporting keys loaded. Check loading logic.")
    except Exception as e:
        print(f"Error configuring Gemini API with initial key {key_names[current_key_index]}: {e}")
        # Potentially try the next key? For now, just warn. The agents will handle retries/switching.
        print("Will attempt configuration again within agent functions.")

# --- LLM Agent Functions using Gemini API (with Multi-Key Switching Retry Logic) ---
# NOTE: Agent functions (agent1_translate_gemini, agent2_validate_gemini, agent3_refine_gemini)
# and the switch_gemini_key function remain unchanged.
# They will use the keys loaded into the global 'api_keys' list.

def switch_gemini_key(agent_name: str) -> bool:
    """
    Attempts to switch to the next available Gemini API key.

    Args:
        agent_name: Name of the agent requesting the switch (for logging).

    Returns:
        True if key switch was successful, False otherwise.
    """
    global current_key_index, api_keys, key_names

    if len(api_keys) <= 1:
        print(f"{agent_name}: No other keys available to switch to.")
        return False # Cannot switch if only 0 or 1 key exists

    initial_index = current_key_index
    next_key_index = (current_key_index + 1) % len(api_keys)
    print(f"{agent_name}: Attempting to switch from {key_names[current_key_index]} to {key_names[next_key_index]}.")

    try:
        genai.configure(api_key=api_keys[next_key_index])
        current_key_index = next_key_index # Update global index only on success
        print(f"{agent_name}: Successfully switched to API key: {key_names[current_key_index]}.")
        # Re-initialize the model instance after switching key in the agent function
        return True
    except Exception as config_e:
        print(f"{agent_name} Error: Failed to configure/re-init with key {key_names[next_key_index]}: {config_e}.")
        # Do not change current_key_index if switch failed
        print(f"{agent_name}: Continuing with the current key: {key_names[initial_index]}.")
        return False


def agent1_translate_gemini(english_text: str) -> str:
    """
    Uses Agent-1 (Gemini API) for translation with multi-key retry logic.
    """
    global current_key_index, api_keys, key_names, gemini_model_name
    agent_id = "Agent 1" # For logging

    # print(f"--- {agent_id} (Gemini API - {gemini_model_name}): Translating to Arabic ---") # Reduced verbosity inside loop
    if not api_keys:
        print(f"{agent_id} Error: No API keys loaded.")
        return f"###ERROR: {agent_id} - Translation failed (No API Keys)###"

    # Define the UPDATED prompt for the translation task.
    prompt = f"""You are an AI assistant whose job is to translate some given questions from English to Arabic. While translation, there will be tags starting with ### signs (like ###Question). Do not translate these tags. Make sure you translate all the tag contents. Please follow the guidelines:
1. Maintain Meaning: Ensure the translated question conveys the original intent.
2. Cultural Adaptation: Adjust cultural references to suit the target language. For example, adapt place names, idioms, festivals, toys, objects and cultural symbols as needed (e.g., ’Louvre Museum’ should be localized appropriately).
3. Context Sensitivity: Choose translations that match the context, avoiding direct word-for-word translations that may distort meaning.
4. Natural Expression: Ensure the translation flows naturally in Arabic, preserving the readability and coherence.
5. There will be options with A,B,C or D. Do not translate the options letters and keep them in the same order.
6. Make sure all the elements are present in your response, like ###STORY, ###QUESTION and ###OPTIONS.

Input Text (English):
---
{english_text}
---

Output Translation (Arabic):"""

    retries_on_current_key = 0
    total_attempts = 0
    # Max attempts slightly adjusted for clarity, actual limit is the while loop condition
    max_total_attempts = MAX_RETRIES_PER_KEY * len(api_keys) + len(api_keys) # Allow one initial try per key
    should_retry = False # Initialize should_retry flag

    while total_attempts < max_total_attempts:
        # Ensure current_key_index is valid before accessing lists
        if current_key_index >= len(key_names) or current_key_index >= len(api_keys):
             print(f"{agent_id} Error: Invalid current_key_index ({current_key_index}) for api_keys list (size {len(api_keys)}).")
             return f"###ERROR: {agent_id} - Internal key index error.###"

        active_key_for_attempt = key_names[current_key_index]
        # print(f"{agent_id}: Attempting API call (Total Attempt {total_attempts + 1}, Key: {active_key_for_attempt}, Retry on this key: {retries_on_current_key})") # Reduced verbosity

        try:
            # --- Model Initialization (Inside Loop for Key Switching) ---
            # Ensure genai is configured with the *correct* key for this attempt
            # This might be redundant if switch_gemini_key worked, but safer
            genai.configure(api_key=api_keys[current_key_index])
            model = genai.GenerativeModel(gemini_model_name)
            # --- End Model Initialization ---

            time.sleep(0.5 + random.uniform(0, 0.5)) # Slightly reduced base delay + jitter
            request_options = {"timeout": 120} # Request timeout

            # --- API Call ---
            response = model.generate_content(prompt, request_options=request_options)
            # --- End API Call ---

            # --- Process Response ---
            if response.candidates and response.candidates[0].content.parts:
                arabic_translation = response.text
                # print(f"{agent_id}: Translation successful (using key '{active_key_for_attempt}').") # Reduced verbosity
                return arabic_translation # Success!
            else:
                print(f"{agent_id} Error: No content/blocked (using key '{active_key_for_attempt}').")
                finish_reason = response.candidates[0].finish_reason if response.candidates else 'UNKNOWN'
                safety_ratings = response.candidates[0].safety_ratings if response.candidates else 'UNKNOWN'
                print(f"Finish Reason: {finish_reason}, Safety Ratings: {safety_ratings}")
                # Treat block/no content as non-retryable for this agent for now
                return f"###ERROR: {agent_id} - Translation failed (No Content/Blocked - Reason: {finish_reason}, Key: {active_key_for_attempt})###"
            # --- End Process Response ---

        # --- Exception Handling ---
        except (exceptions.ResourceExhausted, exceptions.InternalServerError) as e: # Treat internal server errors (5xx) as potentially retryable too
            error_type = type(e).__name__
            print(f"{agent_id} Caught retryable error ({error_type}) (using key '{active_key_for_attempt}'): {e}")
            should_retry = True
        except exceptions.PermissionDenied as e:
            print(f"{agent_id} Error: API Permission Denied (using key '{active_key_for_attempt}'). Check key/permissions.")
            if len(api_keys) > 1:
                 print(f"{agent_id}: Attempting to switch key due to Permission Denied.")
                 if switch_gemini_key(agent_id):
                     retries_on_current_key = 0 # Reset retry count for the new key
                     total_attempts += 1
                     continue # Go to next iteration with the new key
                 else:
                      return f"###ERROR: {agent_id} - Translation failed (API Permission Denied on {active_key_for_attempt}, switch failed)###"
            else:
                 return f"###ERROR: {agent_id} - Translation failed (API Permission Denied on {active_key_for_attempt})###"
        except exceptions.InvalidArgument as e:
             print(f"{agent_id} Error: Invalid Argument (using key '{active_key_for_attempt}'). Check model/parameters: {e}")
             return f"###ERROR: {agent_id} - Translation failed (Invalid Argument on {active_key_for_attempt})###"
        except Exception as e: # Catch potential configuration errors too
            error_str = str(e).lower()
            if "api_key" in error_str or "configure" in error_str:
                 print(f"{agent_id} Error during API configuration/model init for key {active_key_for_attempt}: {e}")
                 # Treat as permission denied - try switching key
                 if len(api_keys) > 1:
                     print(f"{agent_id}: Attempting to switch key due to configuration error.")
                     if switch_gemini_key(agent_id):
                         retries_on_current_key = 0
                         total_attempts += 1
                         continue
                     else:
                          return f"###ERROR: {agent_id} - Translation failed (Config error on {active_key_for_attempt}, switch failed)###"
                 else:
                     return f"###ERROR: {agent_id} - Translation failed (Config error on {active_key_for_attempt})###"
            elif "429" in error_str and ("quota" in error_str or "resource has been exhausted" in error_str):
                print(f"{agent_id} Caught generic Exception that looks like Quota Error (using key '{active_key_for_attempt}'): {e}")
                should_retry = True
            else:
                print(f"{agent_id} Error: An unexpected non-retryable error occurred (using key '{active_key_for_attempt}'): {type(e).__name__} - {e}")
                return f"###ERROR: {agent_id} - Translation failed ({type(e).__name__} on {active_key_for_attempt})###"
        # --- End Exception Handling ---

        # --- Retry Logic Execution ---
        if should_retry:
            retries_on_current_key += 1
            total_attempts += 1

            if retries_on_current_key > MAX_RETRIES_PER_KEY:
                print(f"{agent_id}: Max retries ({MAX_RETRIES_PER_KEY}) reached for key {active_key_for_attempt}.")
                if len(api_keys) > 1:
                    print(f"{agent_id}: Attempting to switch to the next key.")
                    if switch_gemini_key(agent_id):
                        retries_on_current_key = 0 # Reset counter for the new key
                        # Continue to the next iteration of the while loop
                    else:
                        print(f"{agent_id} Error: Failed to switch key after exhausting retries on {active_key_for_attempt}.")
                        return f"###ERROR: {agent_id} - Translation failed (Quota Exceeded on {active_key_for_attempt}, switch failed)###"
                else:
                    print(f"{agent_id} Error: Max retries reached on the only available key.")
                    return f"###ERROR: {agent_id} - Translation failed (Quota Exceeded after retries on {active_key_for_attempt})###"
            else:
                delay = INITIAL_BACKOFF_DELAY * (BACKOFF_FACTOR ** (retries_on_current_key - 1)) + random.uniform(0, 1)
                print(f"{agent_id} Warning: Retryable error. Retrying in {delay:.2f} seconds... (Attempt {retries_on_current_key}/{MAX_RETRIES_PER_KEY} on key {active_key_for_attempt})")
                time.sleep(delay)

        should_retry = False # Reset for next potential error
        # --- End Retry Logic Execution ---

    # --- Fallback if max total attempts reached ---
    print(f"{agent_id} Error: Maximum total attempts ({max_total_attempts}) reached across all keys.")
    # Ensure index is valid before final error message
    last_key_name = key_names[current_key_index] if current_key_index < len(key_names) else "Invalid Index"
    return f"###ERROR: {agent_id} - Translation failed (Max total attempts reached, last key: {last_key_name})###"


def agent2_validate_gemini(original_english: str, initial_arabic: str) -> str:
    """
    Uses Agent-2 (Gemini API) for validation with multi-key retry logic.
    """
    global current_key_index, api_keys, key_names, gemini_model_name
    agent_id = "Agent 2" # For logging

    # print(f"--- {agent_id} (Gemini API - {gemini_model_name}): Validating Translation ---") # Reduced verbosity
    if not api_keys:
        print(f"{agent_id} Error: No API keys loaded.")
        return f"###ERROR: {agent_id} - Validation failed (No API Keys)###"

    # Define the UPDATED prompt for the validation task.
    prompt = f"""You are tasked with verifying a translation from English to Arabic. You will be given the original text and the translated text. Your job is to check the following:
1. Accuracy of Meaning: Ensure that the translated text preserves the same meaning as the original. Point out any inconsistencies or loss of information.
2. Cultural Adaptation: Verify if any cultural references or context-specific terms are correctly translated, maintaining cultural appropriateness for Arabic.
3. Contextual Relevance: Check if the translation uses contextually correct words or phrases, ensuring that any ambiguities or multiple meanings are handled properly.
4. Suggestions: Provide constructive feedback on how to improve the translation, if necessary, in English.

Your goal is to ensure that the translation is accurate, natural, and culturally appropriate. Your task is to return two tags in your output:
###Quality: respond with okay if the translation looks good.
###Feedback: Only respond with feedback if the translation is not good. Put this tag before the feedback.

Example 1 (Good Translation):
###Quality: okay

Example 2 (Needs Improvement):
###Quality: not okay
###Feedback: The translation is not good. The aspect of location of the translation is missing.

Do not comment on the tags(###) inside the original or translated text provided below.

Original Text (English):
---
{original_english}
---

Translated Text (Arabic):
---
{initial_arabic}
---

Verification Output:"""

    # --- Retry Loop Logic (Similar to Agent 1) ---
    retries_on_current_key = 0
    total_attempts = 0
    max_total_attempts = MAX_RETRIES_PER_KEY * len(api_keys) + len(api_keys)
    should_retry = False # Initialize should_retry flag

    while total_attempts < max_total_attempts:
        # Ensure current_key_index is valid before accessing lists
        if current_key_index >= len(key_names) or current_key_index >= len(api_keys):
             print(f"{agent_id} Error: Invalid current_key_index ({current_key_index}) for api_keys list (size {len(api_keys)}).")
             return f"###ERROR: {agent_id} - Internal key index error.###"

        active_key_for_attempt = key_names[current_key_index]
        # print(f"{agent_id}: Attempting API call (Total Attempt {total_attempts + 1}, Key: {active_key_for_attempt}, Retry on this key: {retries_on_current_key})") # Reduced verbosity

        try:
            # Ensure genai is configured with the *correct* key for this attempt
            genai.configure(api_key=api_keys[current_key_index])
            model = genai.GenerativeModel(gemini_model_name)
            time.sleep(0.5 + random.uniform(0, 0.5))
            request_options = {"timeout": 120}
            response = model.generate_content(prompt, request_options=request_options)

            if response.candidates and response.candidates[0].content.parts:
                feedback_text = response.text
                if feedback_text and feedback_text.strip():
                    # print(f"{agent_id}: Validation feedback generated successfully (using key '{active_key_for_attempt}').") # Reduced verbosity
                    if "###Quality:" not in feedback_text:
                         print(f"{agent_id} Warning: Output missing '###Quality:' tag (Key: {active_key_for_attempt}).")
                    return feedback_text
                else:
                    print(f"{agent_id} Error: Received empty feedback (using key '{active_key_for_attempt}'). Retrying...")
                    should_retry = True # Treat empty response as retryable
            else:
                print(f"{agent_id} Error: No content/blocked (using key '{active_key_for_attempt}').")
                finish_reason = response.candidates[0].finish_reason if response.candidates else 'UNKNOWN'
                # Treat block/no content as non-retryable for this agent
                return f"###ERROR: {agent_id} - Validation failed (No Content/Blocked - Reason: {finish_reason}, Key: {active_key_for_attempt})###"

        except (exceptions.ResourceExhausted, exceptions.InternalServerError) as e:
            error_type = type(e).__name__
            print(f"{agent_id} Caught retryable error ({error_type}) (using key '{active_key_for_attempt}'): {e}")
            should_retry = True
        except exceptions.PermissionDenied as e:
            print(f"{agent_id} Error: API Permission Denied on {active_key_for_attempt}: {e}")
            if len(api_keys) > 1:
                 print(f"{agent_id}: Attempting key switch.")
                 if switch_gemini_key(agent_id):
                     retries_on_current_key = 0
                     total_attempts += 1
                     continue
                 else:
                      return f"###ERROR: {agent_id} - Validation failed (Permission Denied on {active_key_for_attempt}, switch failed)###"
            else:
                 return f"###ERROR: {agent_id} - Validation failed (Permission Denied on {active_key_for_attempt})###"
        except exceptions.InvalidArgument as e:
             print(f"{agent_id} Error: Invalid Argument on {active_key_for_attempt}: {e}")
             return f"###ERROR: {agent_id} - Validation failed (Invalid Argument on {active_key_for_attempt})###"
        except Exception as e: # Catch potential configuration errors too
            error_str = str(e).lower()
            if "api_key" in error_str or "configure" in error_str:
                 print(f"{agent_id} Error during API configuration/model init for key {active_key_for_attempt}: {e}")
                 if len(api_keys) > 1:
                     print(f"{agent_id}: Attempting to switch key due to configuration error.")
                     if switch_gemini_key(agent_id):
                         retries_on_current_key = 0
                         total_attempts += 1
                         continue
                     else:
                          return f"###ERROR: {agent_id} - Validation failed (Config error on {active_key_for_attempt}, switch failed)###"
                 else:
                     return f"###ERROR: {agent_id} - Validation failed (Config error on {active_key_for_attempt})###"
            elif "429" in error_str and ("quota" in error_str or "resource has been exhausted" in error_str):
                print(f"{agent_id} Caught generic Quota Error on {active_key_for_attempt}: {e}")
                should_retry = True
            else:
                print(f"{agent_id} Error: Unexpected non-retryable error on {active_key_for_attempt}: {type(e).__name__} - {e}")
                return f"###ERROR: {agent_id} - Validation failed ({type(e).__name__} on {active_key_for_attempt})###"

        if should_retry:
            retries_on_current_key += 1
            total_attempts += 1

            if retries_on_current_key > MAX_RETRIES_PER_KEY:
                print(f"{agent_id}: Max retries reached for key {active_key_for_attempt}.")
                if len(api_keys) > 1:
                    print(f"{agent_id}: Attempting key switch.")
                    if switch_gemini_key(agent_id):
                        retries_on_current_key = 0
                    else:
                        print(f"{agent_id} Error: Key switch failed after exhausting retries on {active_key_for_attempt}.")
                        return f"###ERROR: {agent_id} - Validation failed (Quota Exceeded on {active_key_for_attempt}, switch failed)###"
                else:
                    print(f"{agent_id} Error: Max retries reached on the only key.")
                    return f"###ERROR: {agent_id} - Validation failed (Quota Exceeded on {active_key_for_attempt})###"
            else:
                delay = INITIAL_BACKOFF_DELAY * (BACKOFF_FACTOR ** (retries_on_current_key - 1)) + random.uniform(0, 1)
                print(f"{agent_id} Warning: Retrying in {delay:.2f}s (Attempt {retries_on_current_key}/{MAX_RETRIES_PER_KEY} on key {active_key_for_attempt})")
                time.sleep(delay)

        should_retry = False
    # --- End Retry Loop ---

    print(f"{agent_id} Error: Maximum total attempts ({max_total_attempts}) reached.")
    last_key_name = key_names[current_key_index] if current_key_index < len(key_names) else "Invalid Index"
    return f"###ERROR: {agent_id} - Validation failed (Max total attempts reached, last key: {last_key_name})###"


def agent3_refine_gemini(original_english: str, initial_arabic: str, feedback: str) -> str:
    """
    Uses Agent-3 (Gemini API) for refinement with multi-key retry logic.
    """
    global current_key_index, api_keys, key_names, gemini_model_name
    agent_id = "Agent 3" # For logging

    # print(f"--- {agent_id} (Gemini API - {gemini_model_name}): Refining Translation ---") # Reduced verbosity
    if not api_keys:
        print(f"{agent_id} Error: No API keys loaded.")
        return initial_arabic + f"\n###ERROR: {agent_id} - Refinement failed (No API Keys)###"

    # --- Feedback Processing (Same as before) ---
    if feedback.startswith("###ERROR:"):
        # print(f"{agent_id} Skipping: Received error feedback string from Agent 2: {feedback}") # Reduced verbosity
        return initial_arabic + f"\n###ERROR: {agent_id} - Skipped due to Agent 2 error ({feedback})###"
    else:
        feedback_prompt_part = f"Feedback from Verification Agent:\n---\n{feedback}\n---"
    # --- End Feedback Processing ---

    # Define the UPDATED prompt for the refinement task.
    prompt = f"""You are an AI assistant who is capable of translating from English to Arabic.
You will be given an English text and an initial translation into Arabic.
However, there may be problems in the initial translation, such as missed incidents, tweaked details, or other inaccuracies.
You will also be given feedback from a verification agent. Closely follow the feedback provided (especially the text after the '###Feedback:' tag, if present) to improve the translation or insert any missing elements in the story, question, or options section.
Ensure all structural tags (###STORY, ###QUESTION, ###OPTIONS, including option markers like (A), (B)) are present, correctly formatted, and match the structure of the original English text.
Output ONLY the complete, refined Arabic text, including all necessary tags. Do not add any explanations or introductory phrases.

Original English Text:
---
{original_english}
---

Initial Arabic Translation:
---
{initial_arabic}
---

{feedback_prompt_part}

Refined Arabic Translation (including all tags):"""

    # --- Retry Loop Logic (Similar to Agent 1 & 2) ---
    retries_on_current_key = 0
    total_attempts = 0
    max_total_attempts = MAX_RETRIES_PER_KEY * len(api_keys) + len(api_keys)
    should_retry = False # Initialize should_retry flag

    while total_attempts < max_total_attempts:
        # Ensure current_key_index is valid before accessing lists
        if current_key_index >= len(key_names) or current_key_index >= len(api_keys):
             print(f"{agent_id} Error: Invalid current_key_index ({current_key_index}) for api_keys list (size {len(api_keys)}).")
             return initial_arabic + f"\n###ERROR: {agent_id} - Internal key index error.###"

        active_key_for_attempt = key_names[current_key_index]
        # print(f"{agent_id}: Attempting API call (Total Attempt {total_attempts + 1}, Key: {active_key_for_attempt}, Retry on this key: {retries_on_current_key})") # Reduced verbosity

        try:
            # Ensure genai is configured with the *correct* key for this attempt
            genai.configure(api_key=api_keys[current_key_index])
            model = genai.GenerativeModel(gemini_model_name)
            time.sleep(0.5 + random.uniform(0, 0.5))
            request_options = {"timeout": 120}
            response = model.generate_content(prompt, request_options=request_options)

            if response.candidates and response.candidates[0].content.parts:
                refined_arabic = response.text
                # --- Basic Sanity Check (Same as before) ---
                if refined_arabic and "###STORY" in refined_arabic and "###QUESTION" in refined_arabic and len(refined_arabic) > len(initial_arabic) * 0.5 :
                    # print(f"{agent_id}: Refinement successful (using key '{active_key_for_attempt}').") # Reduced verbosity
                    return refined_arabic
                else:
                    print(f"{agent_id} Warning: Refined text seems invalid (using key '{active_key_for_attempt}'). Retrying...")
                    # Treat invalid output as potentially retryable
                    should_retry = True
                    # return initial_arabic + f"\n###WARNING: {agent_id} - Refinement produced potentially invalid output (Key: {active_key_for_attempt})###"
            else:
                print(f"{agent_id} Error: No content/blocked (using key '{active_key_for_attempt}').")
                finish_reason = response.candidates[0].finish_reason if response.candidates else 'UNKNOWN'
                # Treat block/no content as non-retryable
                return initial_arabic + f"\n###ERROR: {agent_id} - Refinement failed (No Content/Blocked - Reason: {finish_reason}, Key: {active_key_for_attempt})###"

        except (exceptions.ResourceExhausted, exceptions.InternalServerError) as e:
            error_type = type(e).__name__
            print(f"{agent_id} Caught retryable error ({error_type}) (using key '{active_key_for_attempt}'): {e}")
            should_retry = True
        except exceptions.PermissionDenied as e:
            print(f"{agent_id} Error: API Permission Denied on {active_key_for_attempt}: {e}")
            if len(api_keys) > 1:
                 print(f"{agent_id}: Attempting key switch.")
                 if switch_gemini_key(agent_id):
                     retries_on_current_key = 0
                     total_attempts += 1
                     continue
                 else:
                      return initial_arabic + f"\n###ERROR: {agent_id} - Refinement failed (Permission Denied on {active_key_for_attempt}, switch failed)###"
            else:
                 return initial_arabic + f"\n###ERROR: {agent_id} - Refinement failed (Permission Denied on {active_key_for_attempt})###"
        except exceptions.InvalidArgument as e:
             print(f"{agent_id} Error: Invalid Argument on {active_key_for_attempt}: {e}")
             return initial_arabic + f"\n###ERROR: {agent_id} - Refinement failed (Invalid Argument on {active_key_for_attempt})###"
        except Exception as e: # Catch potential configuration errors too
            error_str = str(e).lower()
            if "api_key" in error_str or "configure" in error_str:
                 print(f"{agent_id} Error during API configuration/model init for key {active_key_for_attempt}: {e}")
                 if len(api_keys) > 1:
                     print(f"{agent_id}: Attempting to switch key due to configuration error.")
                     if switch_gemini_key(agent_id):
                         retries_on_current_key = 0
                         total_attempts += 1
                         continue
                     else:
                          return initial_arabic + f"\n###ERROR: {agent_id} - Refinement failed (Config error on {active_key_for_attempt}, switch failed)###"
                 else:
                     return initial_arabic + f"\n###ERROR: {agent_id} - Refinement failed (Config error on {active_key_for_attempt})###"

            elif "429" in error_str and ("quota" in error_str or "resource has been exhausted" in error_str):
                print(f"{agent_id} Caught generic Quota Error on {active_key_for_attempt}: {e}")
                should_retry = True
            else:
                print(f"{agent_id} Error: Unexpected non-retryable error on {active_key_for_attempt}: {type(e).__name__} - {e}")
                return initial_arabic + f"\n###ERROR: {agent_id} - Refinement failed ({type(e).__name__} on {active_key_for_attempt})###"

        if should_retry:
            retries_on_current_key += 1
            total_attempts += 1

            if retries_on_current_key > MAX_RETRIES_PER_KEY:
                print(f"{agent_id}: Max retries reached for key {active_key_for_attempt}.")
                if len(api_keys) > 1:
                    print(f"{agent_id}: Attempting key switch.")
                    if switch_gemini_key(agent_id):
                        retries_on_current_key = 0
                    else:
                        print(f"{agent_id} Error: Key switch failed after exhausting retries on {active_key_for_attempt}.")
                        return initial_arabic + f"\n###ERROR: {agent_id} - Refinement failed (Quota Exceeded on {active_key_for_attempt}, switch failed)###"
                else:
                    print(f"{agent_id} Error: Max retries reached on the only key.")
                    return initial_arabic + f"\n###ERROR: {agent_id} - Refinement failed (Quota Exceeded on {active_key_for_attempt})###"
            else:
                delay = INITIAL_BACKOFF_DELAY * (BACKOFF_FACTOR ** (retries_on_current_key - 1)) + random.uniform(0, 1)
                print(f"{agent_id} Warning: Retrying in {delay:.2f}s (Attempt {retries_on_current_key}/{MAX_RETRIES_PER_KEY} on key {active_key_for_attempt})")
                time.sleep(delay)

        should_retry = False
    # --- End Retry Loop ---

    print(f"{agent_id} Error: Maximum total attempts ({max_total_attempts}) reached.")
    last_key_name = key_names[current_key_index] if current_key_index < len(key_names) else "Invalid Index"
    return initial_arabic + f"\n###ERROR: {agent_id} - Refinement failed (Max total attempts reached, last key: {last_key_name})###"


# --- Main Pipeline Function ---
# MODIFIED to add per-agent timing
def arabic_translation_pipeline(english_sample: dict) -> dict:
    """
    Orchestrates the multi-agent translation pipeline for a single sample using
    the updated prompts and logic. Prints time taken for each agent step.

    Args:
        english_sample: A dictionary containing 'story', 'question', and 'options'.

    Returns:
        A dictionary containing results, including 'feedback' as a raw string.
    """
    results = {}
    pipeline_step_start_time = time.time() # Start timer for this specific pipeline run

    # Prepare Input
    full_english_text = f"###STORY\n{english_sample['story']}\n\n###QUESTION\n{english_sample['question']}\n\n###OPTIONS\n{english_sample['options']}"
    results['original_english'] = full_english_text

    # --- Step 1: Initial Translation ---
    agent1_start_time = time.time()
    initial_arabic_translation = agent1_translate_gemini(full_english_text)
    agent1_end_time = time.time()
    agent1_duration = agent1_end_time - agent1_start_time
    results['initial_arabic'] = initial_arabic_translation
    print(f"    Agent 1 (Translate) completed in: {agent1_duration:.2f} seconds")
    # --- End Step 1 ---

    # --- Step 2: Validation & Feedback ---
    agent2_start_time = time.time()
    if not initial_arabic_translation.startswith("###ERROR:"):
        validation_feedback = agent2_validate_gemini(full_english_text, initial_arabic_translation)
        results['feedback'] = validation_feedback
    else:
        results['feedback'] = f"###ERROR: Skipped Agent 2 due to Agent 1 error ({initial_arabic_translation})###"
        results['refined_arabic'] = initial_arabic_translation
        agent2_end_time = time.time() # Still record time even if skipped
        agent2_duration = agent2_end_time - agent2_start_time
        print(f"    Agent 2 (Validate) skipped due to Agent 1 error ({agent2_duration:.2f} seconds)")
        pipeline_end_time = time.time()
        total_pipeline_duration = pipeline_end_time - pipeline_step_start_time
        print(f"    -- Pipeline for this line took: {total_pipeline_duration:.2f} seconds --")
        return results # Exit early

    agent2_end_time = time.time()
    agent2_duration = agent2_end_time - agent2_start_time
    print(f"    Agent 2 (Validate) completed in: {agent2_duration:.2f} seconds")
    # --- End Step 2 ---

    # --- Step 3: Refinement ---
    agent3_start_time = time.time()
    refined_arabic_translation = agent3_refine_gemini(full_english_text, initial_arabic_translation, results['feedback'])
    agent3_end_time = time.time()
    agent3_duration = agent3_end_time - agent3_start_time
    results['refined_arabic'] = refined_arabic_translation
    print(f"    Agent 3 (Refine) completed in: {agent3_duration:.2f} seconds")
    # --- End Step 3 ---

    pipeline_end_time = time.time()
    total_pipeline_duration = pipeline_end_time - pipeline_step_start_time
    print(f"    -- Pipeline for this line took: {total_pipeline_duration:.2f} seconds --")

    return results


# --- Data Loading and Processing ---
# ADDED ETA Calculation
def load_and_process_data(data_dir: str):
    """
    Loads data from .jsonl files, processes each line through the pipeline,
    collects results, and provides ETA updates.

    Args:
        data_dir: The path to the directory containing the .jsonl data files.

    Returns:
        A list of dictionaries with results for each processed line.
    """
    print(f"\n--- Loading and Processing Data from Directory: {data_dir} ---")
    processed_results = []
    file_count = 0
    total_lines_attempted = 0
    lines_successfully_parsed = 0
    lines_skipped_or_failed_parsing = 0
    pipeline_error_count = 0
    total_lines_to_process = 0 # For ETA

    if not os.path.isdir(data_dir):
        print(f"Error: Data directory '{data_dir}' not found or is not a directory.")
        return processed_results

    # --- First Pass: Count total lines for ETA ---
    print("--- Counting total lines for ETA estimation ---")
    for filename in os.listdir(data_dir):
        file_path = os.path.join(data_dir, filename)
        if filename.endswith(".jsonl") and os.path.isfile(file_path):
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        if line.strip(): # Count only non-empty lines
                            total_lines_to_process += 1
            except Exception as e:
                print(f"Warning: Could not read file {filename} during count: {e}")
    print(f"--- Found {total_lines_to_process} non-empty lines to process across all .jsonl files ---")
    # --- End First Pass ---

    if total_lines_to_process == 0:
        print("No lines found to process. Exiting.")
        return processed_results

    # --- Second Pass: Process data ---
    start_time = time.time() # Record start time for ETA
    processed_lines_count = 0 # Track processed lines for ETA

    for filename in os.listdir(data_dir):
        file_path = os.path.join(data_dir, filename)

        if filename.endswith(".jsonl") and os.path.isfile(file_path):
            file_count += 1
            print(f"\n--- Processing file: {filename} ---")
            lines_in_file = 0
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    for i, line in enumerate(f):
                        line_num = i + 1
                        total_lines_attempted += 1
                        lines_in_file += 1
                        print(f"  Processing line {line_num}/{lines_in_file} in {filename} (Overall: {processed_lines_count+1}/{total_lines_to_process})...") # More detailed progress

                        if not line.strip():
                            # print(f"    Skipping empty line {line_num}.") # Reduced verbosity
                            lines_skipped_or_failed_parsing += 1
                            continue

                        try:
                            data = json.loads(line)
                            lines_successfully_parsed += 1

                            # --- Extract and Validate Required Data ---
                            data_upper = {k.upper(): v for k, v in data.items()}
                            story_text = data_upper.get("STORY")
                            question_text = data_upper.get("QUESTION")
                            option_keys = sorted([k for k in data if k.upper().startswith("OPTION-")])

                            missing_keys = []
                            if not story_text: missing_keys.append("STORY")
                            if not question_text: missing_keys.append("QUESTION")
                            if not option_keys: missing_keys.append("OPTION-*")

                            if missing_keys:
                                print(f"    Warning: Skipping line {line_num} in {filename} due to missing keys: {', '.join(missing_keys)}.")
                                lines_skipped_or_failed_parsing += 1
                                continue
                            # --- End Extract and Validate ---

                            # --- Prepare Options String ---
                            options_string = ""
                            option_values = [data[k] for k in option_keys]
                            if option_values:
                                options_string = "\n".join(
                                    [f"({chr(65+j)}) {opt}" for j, opt in enumerate(option_values)]
                                )
                            # --- End Prepare Options String ---

                            # --- Prepare Input for Pipeline ---
                            english_sample = {
                                "story": story_text,
                                "question": question_text,
                                "options": options_string
                            }
                            # --- End Prepare Input ---

                            # --- Run the Updated Translation Pipeline ---
                            pipeline_output = arabic_translation_pipeline(english_sample)
                            # --- End Run Pipeline ---

                            # --- Store Results ---
                            result_entry = {
                                "source_file": filename,
                                "line_number": line_num,
                                # FIX: Use raw string r"..." for the key containing \N
                                "scenario_id": data_upper.get(r"序号\NINDEX", data_upper.get("INDEX", data_upper.get("ID", f"{filename}_line{line_num}"))),
                                **pipeline_output
                            }
                            processed_results.append(result_entry)
                            processed_lines_count += 1 # Increment successfully processed count for ETA
                            # --- End Store Results ---

                            # --- Check for Pipeline Errors ---
                            if "###ERROR:" in pipeline_output.get('refined_arabic', '') or \
                               "###WARNING:" in pipeline_output.get('refined_arabic', ''):
                                # print(f"    Pipeline completed with errors/warnings for line {line_num}.") # Reduced verbosity
                                pipeline_error_count += 1
                            # else:
                                # print(f"    Successfully processed line {line_num}.") # Reduced verbosity
                            # --- End Check for Pipeline Errors ---

                            # --- ETA Calculation and Update ---
                            if processed_lines_count >= MIN_LINES_FOR_ETA and processed_lines_count % ETA_UPDATE_INTERVAL_LINES == 0:
                                current_time = time.time()
                                elapsed_time = current_time - start_time
                                avg_time_per_line = elapsed_time / processed_lines_count
                                remaining_lines = total_lines_to_process - processed_lines_count
                                if remaining_lines > 0: # Avoid division by zero or negative time if already finished
                                     eta_seconds = avg_time_per_line * remaining_lines
                                     eta_formatted = str(timedelta(seconds=int(eta_seconds))) # Format as H:MM:SS
                                else:
                                     eta_formatted = "0:00:00"
                                progress_percent = (processed_lines_count / total_lines_to_process) * 100
                                print(f"  Progress: {processed_lines_count}/{total_lines_to_process} lines ({progress_percent:.1f}%) processed. Avg time/line: {avg_time_per_line:.2f}s. ETA: {eta_formatted}")
                            # --- End ETA ---


                        except json.JSONDecodeError:
                            print(f"    Error: Could not decode JSON from line {line_num} in {filename}. Skipping.")
                            lines_skipped_or_failed_parsing += 1
                        except Exception as e:
                            print(f"    Error processing line {line_num} in {filename}: {type(e).__name__} - {e}. Skipping.")
                            lines_skipped_or_failed_parsing += 1

                # print(f"--- Finished processing file: {filename} ({lines_in_file} lines attempted) ---") # Reduced verbosity

            except FileNotFoundError:
                print(f"Error: File not found: {file_path}")
            except Exception as e:
                print(f"Error reading file {filename}: {type(e).__name__} - {e}")

        elif os.path.isdir(file_path):
             print(f"Skipping directory: {filename}")
        elif not filename.endswith(".jsonl"):
             print(f"Skipping non-JSONL file: {filename}")
    # --- End Second Pass ---

    # --- Final Summary ---
    end_time = time.time()
    total_processing_time = end_time - start_time
    total_time_formatted = str(timedelta(seconds=int(total_processing_time)))

    print(f"\n--- Finished processing all files in directory: {data_dir} ---")
    print(f"Total processing time: {total_time_formatted}")
    print(f"Total .jsonl files found: {file_count}")
    print(f"Total non-empty lines found: {total_lines_to_process}")
    # print(f"Total lines attempted: {total_lines_attempted}") # Can be slightly different if files change between passes
    print(f"Lines successfully parsed as JSON: {lines_successfully_parsed}")
    print(f"Lines skipped or failed parsing/validation: {lines_skipped_or_failed_parsing}")
    print(f"Lines processed through pipeline (input valid): {len(processed_results)}") # This is processed_lines_count
    print(f"Lines completed with pipeline errors/warnings in final output: {pipeline_error_count}")
    # --- End Final Summary ---

    return processed_results


# --- Main Execution Block ---
if __name__ == "__main__":
    # --- Pre-run Check ---
    if not api_keys:
         print("\nCRITICAL ERROR: No API keys found in Colab Secrets or OS Environment (checked GEMINI_API_KEY_1 to GEMINI_API_KEY_6).")
         print("Please add at least one key and try again.")
         print("Script cannot proceed.")
    else:
        # --- Define Data and Output Paths ---
        actual_data_dir = "./data" # MODIFY THIS if your data is elsewhere
        output_file_path = "translation_results_pipeline_multi_key_eta_colab.jsonl" # Output filename
        try:
            # Check if running in Colab and drive is mounted
            from google.colab import drive
            # drive.mount('/content/drive', force_remount=True) # Mount is now done earlier
            output_file_path = '/content/drive/My Drive/translation_results_pipeline_multi_key_eta_colab.jsonl' # Output filename
            print(f"Output path set to Google Drive: {output_file_path}")
        except ImportError:
            # Not in Colab or drive mount failed
            print(f"Output path set to local directory: {output_file_path}")
        # --- End Define Paths ---

        # --- Run Data Processing ---
        all_final_results = load_and_process_data(actual_data_dir)
        # --- End Run Data Processing ---

        # --- Display Example Result ---
        if all_final_results:
            print("\n--- Example Result (from first processed line) ---")
            # Avoid error if list is somehow empty after processing
            try:
                first_result = all_final_results[0]
                print(f"Source File: {first_result.get('source_file', 'N/A')}")
                print(f"Line Number: {first_result.get('line_number', 'N/A')}")
                print(f"Scenario ID: {first_result.get('scenario_id', 'N/A')}")
                print(f"\nOriginal English:\n{first_result.get('original_english', 'N/A')}")
                print(f"\nInitial Arabic (Agent 1):\n{first_result.get('initial_arabic', 'N/A')}")

                # Display raw feedback string from Agent 2
                feedback_raw = first_result.get('feedback', 'N/A')
                print(f"\nValidation Feedback (Agent 2 - Raw Text):\n{feedback_raw}")

                print(f"\nRefined Arabic (Agent 3 - Ready for Human Review):\n{first_result.get('refined_arabic', 'N/A')}")
            except IndexError:
                print("Processing seemed complete, but no results found in the list to display.")
        else:
            print("\nProcessing completed, but no results were generated.")
            print("Check data directory, file format (.jsonl), content (required keys), and API key setup.")
        # --- End Display Example Result ---

        # --- Save Results ---
        if all_final_results:
            print(f"\nSaving all {len(all_final_results)} results to {output_file_path}...")
            try:
                output_dir = os.path.dirname(output_file_path)
                if output_dir:
                    os.makedirs(output_dir, exist_ok=True)
                    # print(f"Ensured output directory exists: {output_dir}") # Reduced verbosity

                with open(output_file_path, 'w', encoding='utf-8') as f:
                    for result in all_final_results:
                        json_line = json.dumps(result, ensure_ascii=False)
                        f.write(json_line + '\n')
                print("Results saved successfully.")
            except Exception as e:
                print(f"Error saving results to file {output_file_path}: {type(e).__name__} - {e}")
        else:
            print("\nNo results to save.")
        # --- End Save Results ---

        print("\n--- Script Execution Finished ---")
        # print("Next Step Suggestion: Human review of the 'refined_arabic' outputs, paying attention to Agent 2 feedback.") # Already in summary



Mounted at /content/drive
Google Drive mounted successfully.
Google Colab environment detected. Will try reading secrets from userdata.
--- Loading API Keys ---
Found GEMINI_API_KEY_1 in Colab Secrets.
Found GEMINI_API_KEY_2 in Colab Secrets.
Found GEMINI_API_KEY_3 in Colab Secrets.
Found GEMINI_API_KEY_4 in Colab Secrets.
Found GEMINI_API_KEY_5 in Colab Secrets.
Found GEMINI_API_KEY_6 in Colab Secrets.

Loaded 6 API key(s).
Configuring Gemini API with initial key: Key_1 using model gemini-2.0-flash.
Gemini API configured successfully.
Output path set to Google Drive: /content/drive/My Drive/translation_results_pipeline_multi_key_eta_colab.jsonl

--- Loading and Processing Data from Directory: ./data ---
--- Counting total lines for ETA estimation ---
--- Found 2860 non-empty lines to process across all .jsonl files ---

--- Processing file: combined_output_english.jsonl ---
  Processing line 1/1 in combined_output_english.jsonl (Overall: 1/2860)...
    Agent 1 (Translate) completed in

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 480.67ms


Agent 1 Error: An unexpected non-retryable error occurred (using key 'Key_1'): ReadTimeout - HTTPConnectionPool(host='localhost', port=45855): Read timed out. (read timeout=118.60520195960999)
    Agent 1 (Translate) completed in: 120.72 seconds
    Agent 2 (Validate) skipped due to Agent 1 error (0.00 seconds)
    -- Pipeline for this line took: 120.72 seconds --
  Processing line 4/4 in combined_output_english.jsonl (Overall: 4/2860)...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 137960.94ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 7093.18ms


    Agent 1 (Translate) completed in: 32.22 seconds
    Agent 2 (Validate) completed in: 1.43 seconds
    Agent 3 (Refine) completed in: 6.15 seconds
    -- Pipeline for this line took: 39.80 seconds --
  Processing line 5/5 in combined_output_english.jsonl (Overall: 5/2860)...
    Agent 1 (Translate) completed in: 10.92 seconds
    Agent 2 (Validate) completed in: 4.36 seconds
    Agent 3 (Refine) completed in: 2.49 seconds
    -- Pipeline for this line took: 17.76 seconds --
  Processing line 6/6 in combined_output_english.jsonl (Overall: 6/2860)...


...

Agent 3 Caught generic Quota Error on Key_4: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.
Agent 3 Warning: Retrying in 20.80s (Attempt 2/5 on key Key_4)
    Agent 3 (Refine) completed in: 38.02 seconds
    -- Pipeline for this line took: 42.64 seconds --
  Progress: 1030/2860 lines (36.0%) processed. Avg time/line: 22.44s. ETA: 11:24:28
  Processing line 1031/1031 in combined_output_english.jsonl (Overall: 1031/2860)...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 581.68ms


    Agent 1 (Translate) completed in: 4.32 seconds
    Agent 2 (Validate) completed in: 1.72 seconds
    Agent 3 (Refine) completed in: 2.62 seconds
    -- Pipeline for this line took: 8.67 seconds --
  Processing line 1032/1032 in combined_output_english.jsonl (Overall: 1032/2860)...
    Agent 1 (Translate) completed in: 2.97 seconds
    Agent 2 (Validate) completed in: 6.48 seconds


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 455.23ms


    Agent 3 (Refine) completed in: 3.66 seconds
    -- Pipeline for this line took: 13.11 seconds --
  Processing line 1033/1033 in combined_output_english.jsonl (Overall: 1033/2860)...
    Agent 1 (Translate) completed in: 7.08 seconds
    Agent 2 (Validate) completed in: 1.69 seconds
    Agent 3 (Refine) completed in: 6.17 seconds
    -- Pipeline for this line took: 14.95 seconds --
  Processing line 1034/1034 in combined_output_english.jsonl (Overall: 1034/2860)...
    Agent 1 (Translate) completed in: 3.00 seconds
    Agent 2 (Validate) completed in: 1.59 seconds
    Agent 3 (Refine) completed in: 2.75 seconds
    -- Pipeline for this line took: 7.34 seconds --
  Processing line 1035/1035 in combined_output_english.jsonl (Overall: 1035/2860)...
    Agent 1 (Translate) completed in: 2.68 seconds
    Agent 2 (Validate) completed in: 1.24 seconds
    Agent 3 (Refine) completed in: 2.46 seconds
    -- Pipeline for this line took: 6.38 seconds --
  Processing line 1036/1036 in combined_